# Self-Attention

**Companion lesson:** https://ml-viz-ruby.vercel.app/courses/transformers/01-self-attention

A from-scratch, runnable implementation of the concepts in the lesson.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive. Changes to this view are not saved.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = '#e2e8f0'
plt.rcParams['axes.labelcolor'] = '#e2e8f0'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['axes.edgecolor'] = '#334155'
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.color'] = '#1e293b'
plt.rcParams['figure.figsize'] = (8, 5)
np.random.seed(0)

## Intuition — every token looks at every other token

RNNs pass information step-by-step; **self-attention** lets every token gather information from every
other token *directly*, in one shot. Each token emits three vectors — a **query** (what am I looking
for?), a **key** (what do I contain?), and a **value** (what do I contribute?). Attention scores are
query·key dot products, softmaxed into weights, and each token's output is the weight-averaged values:
`Attention(Q,K,V) = softmax(QKᵀ/√d_k)·V`. The `1/√d_k` keeps the scores' variance at 1 so the softmax
doesn't saturate, and a **causal mask** hides the future for autoregressive models. This one equation
is the engine of every transformer and LLM; we build it from scratch, verify each design choice
numerically, and validate against `jax`.

## Scaled dot-product attention, from scratch

$\text{Attention}(Q,K,V)=\text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right)V$. We implement it, run it on a short sequence, and inspect the attention weights.

In [ ]:
def softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)

def attention(Q, K, V, mask=None):
    d_k = Q.shape[-1]
    scores = Q @ K.swapaxes(-1, -2) / np.sqrt(d_k)
    if mask is not None:
        scores = np.where(mask, scores, -1e9)
    weights = softmax(scores, axis=-1)
    return weights @ V, weights

class SelfAttention:
    def __init__(self, d_model, d_k, seed=0):
        rng = np.random.RandomState(seed)
        self.Wq = rng.randn(d_model, d_k)*0.3
        self.Wk = rng.randn(d_model, d_k)*0.3
        self.Wv = rng.randn(d_model, d_k)*0.3
    def __call__(self, X, mask=None):
        return attention(X@self.Wq, X@self.Wk, X@self.Wv, mask)

X = np.random.randn(5, 8)          # 5 tokens, model dim 8
sa = SelfAttention(d_model=8, d_k=8)
out, W = sa(X)
print('output shape:', out.shape, '| attention rows sum to 1:', np.allclose(W.sum(1), 1))

**What to notice:** the whole mechanism is ~6 lines — scores `Q@Kᵀ/√d_k`, softmax per row, weighted
sum of `V`. Each attention row sums to 1 (a proper distribution over the tokens), and the learned
projections `W_q, W_k, W_v` decide *what* similarity means. Sequence length appears nowhere in the
parameters — attention handles any length with the same weights.

## Worked numeric example, by hand (matches the lesson)

Three tokens, $d_k=2$. Each row of $Q,K,V$ is one token's projected vector:

$$Q=\begin{bmatrix}2&0\\0&2\\2&2\end{bmatrix},\quad K=\begin{bmatrix}2&0\\0&2\\1&1\end{bmatrix},\quad V=\begin{bmatrix}1&0\\0&1\\10&10\end{bmatrix}.$$

1. **Scores** $QK^\top=\begin{bmatrix}4&0&2\\0&4&2\\4&4&4\end{bmatrix}$ (entry $(i,j)=\mathbf q_i\cdot\mathbf k_j$).
2. **Scale** by $1/\sqrt2\approx0.707$.
3. **Softmax** each row, e.g. row 1: $e^{2.828},e^{0},e^{1.414}=16.92,1,4.11$, sum $22.03\Rightarrow[0.768,0.045,0.187]$. Row 3's scores are all equal, so it is exactly uniform $[\tfrac13,\tfrac13,\tfrac13]$.
4. **Weighted sum** of value rows.

The cell below recomputes everything in **pure Python (no numpy)** so every number is deterministic and hand-checkable, then asserts the known answers.

In [ ]:
import math  # pure stdlib: deterministic, hand-checkable

def softmax_row(row):
    e = [math.exp(s) for s in row]
    z = sum(e)
    return [x / z for x in e]

Q = [[2, 0], [0, 2], [2, 2]]
K = [[2, 0], [0, 2], [1, 1]]
V = [[1, 0], [0, 1], [10, 10]]
d_k = 2

# Step 1: scores QKt  -> Step 2: scale -> Step 3: softmax -> Step 4: @V
scores  = [[sum(q[d] * k[d] for d in range(d_k)) for k in K] for q in Q]
scaled  = [[s / math.sqrt(d_k) for s in row] for row in scores]
weights = [softmax_row(row) for row in scaled]
out = [[sum(weights[i][j] * V[j][d] for j in range(3)) for d in range(d_k)]
       for i in range(3)]

print("QKt      :", scores)
print("weights  :", [[round(w, 3) for w in r] for r in weights])
print("row sums :", [round(sum(r), 6) for r in weights])
print("output   :", [[round(o, 3) for o in r] for r in out])

# Deterministic verification against the by-hand answers in the lesson
assert scores == [[4, 0, 2], [0, 4, 2], [4, 4, 4]]
assert all(abs(sum(r) - 1.0) < 1e-12 for r in weights)             # softmax rows normalize
assert [round(w, 3) for w in weights[0]] == [0.768, 0.045, 0.187]  # token 1 attends to key 1
assert [round(w, 3) for w in weights[2]] == [0.333, 0.333, 0.333]  # token 3 is uniform
assert [round(o, 3) for o in out[0]] == [2.635, 1.912]
assert [round(o, 3) for o in out[2]] == [3.667, 3.667]             # uniform -> mean of values
print()
print("All hand-checked values verified.")

**What to notice:** the pure-Python trace matches the lesson's hand computation number for number —
the query·key scores, the softmax weights, and the final blended value. Attention isn't a black box;
it's three matrix multiplies and a softmax you can verify by hand.

## The library way — validate against `jax`

The reference check: reimplement scaled dot-product attention with `jax.nn.softmax` and `einsum`
(exactly what `torch.nn.functional.scaled_dot_product_attention` computes, minus the fused kernels)
and assert outputs and weights match ours.

In [ ]:
import jax, jax.numpy as jnp

def attention_jax(Q, K, V):
    d_k = Q.shape[-1]
    scores = jnp.einsum('qd,kd->qk', jnp.asarray(Q), jnp.asarray(K)) / jnp.sqrt(d_k)
    w = jax.nn.softmax(scores, axis=-1)
    return w @ jnp.asarray(V), w

Qm, Km, Vm = X @ sa.Wq, X @ sa.Wk, X @ sa.Wv
out_np, W_np = attention(Qm, Km, Vm)
out_jx, W_jx = attention_jax(Qm, Km, Vm)
assert np.allclose(out_np, np.array(out_jx), atol=1e-6), "outputs must match jax"
assert np.allclose(W_np, np.array(W_jx), atol=1e-6), "attention weights must match jax"
print('our attention == jax (einsum + jax.nn.softmax) == F.scaled_dot_product_attention math ✓')

**What to notice:** exact agreement — our NumPy attention *is* the operation inside every
transformer. Frameworks add fused FlashAttention kernels for speed and memory, but the math is these
four lines.

## Why divide by $\sqrt{d_k}$? — verify the variance

For unit-variance $\mathbf q,\mathbf k$, the dot product $\mathbf q\cdot\mathbf k=\sum_{i=1}^{d_k}q_ik_i$ is a sum of $d_k$ independent terms each with variance $1$, so $\operatorname{Var}(\mathbf q\cdot\mathbf k)=d_k$. Scores therefore scale like $\sqrt{d_k}$ — without correction, large $d_k$ pushes softmax into its saturated, near-zero-gradient regime. Dividing by $\sqrt{d_k}$ restores variance $1$. The cell below confirms $\operatorname{Var}\approx d_k$ empirically using only the stdlib `random` module (deterministic via a fixed seed).

In [ ]:
import random

def dot_product_variance(d_k, trials=20000, seed=1):
    rng = random.Random(seed)
    vals = []
    for _ in range(trials):
        q = [rng.gauss(0, 1) for _ in range(d_k)]
        k = [rng.gauss(0, 1) for _ in range(d_k)]
        vals.append(sum(qi * ki for qi, ki in zip(q, k)))
    mean = sum(vals) / len(vals)
    return sum((v - mean) ** 2 for v in vals) / len(vals)

for d_k in (2, 8, 64):
    var = dot_product_variance(d_k)
    # scaling by 1/sqrt(d_k) divides the variance by d_k -> ~1
    print("d_k=%2d: Var(q.k)=%6.2f  (theory %d)   scaled Var=%.3f"
          % (d_k, var, d_k, var / d_k))

# unscaled variance grows with d_k; the 1/sqrt(d_k) scaling pins it near 1
assert abs(dot_product_variance(8) - 8) < 1.0
print()
print("Variance grows as d_k; the sqrt(d_k) scaling keeps it ~1.")

**What to notice:** measured directly, the raw dot product's variance **equals `d_k`** (2, 8, 64…),
and dividing by `√d_k` pins the scaled variance at ~1 for every dimension. Without the scaling, scores
at `d_k=64` would be ~8 standard deviations wide — pushing the softmax into saturation (the gotcha
below). The `√d_k` isn't decoration; it's variance control.

## Visualizing what attends to what

In [ ]:
plt.imshow(W if False else sa(X)[1], cmap='magma')
plt.colorbar(label='attention weight'); plt.xlabel('key token'); plt.ylabel('query token')
plt.title('Self-attention weights'); plt.show()

**What to notice:** the heatmap is the model's *routing table* — row `i` shows where token `i`
gathers its information. With random weights it's diffuse; in a trained model these maps become
interpretable (syntax heads, positional heads), which is why attention visualizations became the
standard lens into transformer behavior.

## Causal masking for autoregressive models

GPT-style generation forbids attending to the future. A lower-triangular mask sets future scores to $-\infty$ so their softmax weight is 0 — note the upper triangle is blank.

In [ ]:
T = 5
mask = np.tril(np.ones((T, T))).astype(bool)   # True = allowed
_, Wc = sa(X, mask=mask)
plt.imshow(Wc, cmap='magma')
plt.title('Causal attention (upper triangle = 0)'); plt.xlabel('key'); plt.ylabel('query')
plt.colorbar(); plt.show()
print('row 0 attends only to token 0:', np.round(Wc[0], 3))

**What to notice:** the **causal mask** zeroes the upper triangle — token `i` can only attend to
positions `≤ i`, and row 0 puts weight 1.0 on itself. This single mask is what turns bidirectional
attention (BERT-style) into an autoregressive language model (GPT-style): same mechanism, one triangular
matrix of difference.

## Gotchas & tradeoffs

- **Attention is O(T²)** in sequence length — every token scores against every other. The cost and
  memory of long contexts is attention's defining limitation (FlashAttention optimizes constants; SSMs
  change the asymptotics).
- **Unscaled scores saturate the softmax.** Large-variance scores make attention one-hot, and the
  softmax gradient vanishes — the failure `√d_k` prevents.
- **Masking must use −∞ (or −1e9) *before* the softmax** — zeroing weights *after* softmax breaks the
  normalization.
- **Permutation-invariant by default.** Attention has no notion of order — without positional
  encodings (next lesson), "dog bites man" and "man bites dog" are identical.

In [ ]:
# 1) Softmax saturation: unscaled large-d_k scores make attention one-hot with vanishing gradients
rng = np.random.default_rng(0)
q, Kbig = rng.standard_normal(64), rng.standard_normal((5, 64))
raw = Kbig @ q
for name, scores in [('unscaled', raw), ('scaled by 1/sqrt(64)', raw / 8.0)]:
    w = np.exp(scores - scores.max()); w /= w.sum()
    print(f'{name:22}: max weight = {w.max():.3f}   entropy = {-(w*np.log(w+1e-12)).sum():.3f}')

# 2) Attention ignores order: permuting the tokens permutes the outputs identically
perm = np.array([3, 1, 4, 0, 2])
out_orig, _ = sa(X)
out_perm, _ = sa(X[perm])
print('\npermuted-input outputs == permuted original outputs?', np.allclose(out_perm, out_orig[perm]))

**What to notice:** the unscaled softmax collapses to near-one-hot (max weight ~1, entropy near 0 —
gradients through it vanish), while the `√d_k`-scaled version stays soft. And permuting the input
tokens just permutes the outputs — attention literally cannot see order, which is why positional
encodings exist (next lesson).

## Key takeaways

- Attention is `softmax(QKᵀ/√dₖ)·V` — a content-based weighted average of value vectors.
- Every token reaches every other in **one** step, fully in parallel.
- A **causal mask** enables autoregressive generation.
- The √dₖ scaling keeps softmax out of its saturated, low-gradient regime.

## ✏️ Your turn

### Exercise 1 — Scaled dot-product attention

The formula is:

$$\\text{Attention}(Q,K,V) = \\text{softmax}\\!\\left(\\frac{QK^\\top}{\\sqrt{d_k}}\\right)V$$

Implement it from scratch (you may use `numpy`). Three properties to verify:
- each row of the weight matrix sums to 1 (softmax)
- identical queries produce uniform attention over all keys
- output shape matches `V`

In [ ]:
import numpy as np

def scaled_dot_product_attention(Q, K, V):
    """Return (output, weights) where output = softmax(Q K^T / sqrt(d_k)) @ V."""
    # TODO(you): implement the three steps — score, softmax, weighted sum
    ...

# Quick smoke-test (do not edit)
Q = np.array([[1.0, 0.0], [0.0, 1.0]])
K = np.array([[1.0, 0.0], [0.0, 1.0], [0.5, 0.5]])
V = np.array([[1.0, 2.0], [3.0, 4.0], [5.0, 6.0]])

In [ ]:
out, W = scaled_dot_product_attention(Q, K, V)

assert W.shape == (2, 3), "weights must be (num_queries, num_keys)"
assert np.allclose(W.sum(axis=1), 1.0, atol=1e-6), \
    "each row of the weight matrix must sum to 1 (softmax normalisation)"

# uniform query–key alignment → uniform weights
Q_same = np.ones((4, 3))
K_same = np.ones((5, 3))
V_same = np.random.randn(5, 2)
_, W_unif = scaled_dot_product_attention(Q_same, K_same, V_same)
assert np.allclose(W_unif, 1.0 / 5, atol=1e-6), \
    "when all dot-products are equal, softmax must be uniform (1/num_keys)"

assert out.shape == (Q.shape[0], V.shape[1]), "output shape must be (num_queries, d_v)"

# Edge case: single-token sequence (T=1) -- the only key is itself, weight must be exactly 1
Q1 = np.array([[3.0, -2.0]])
K1 = np.array([[3.0, -2.0]])
V1 = np.array([[7.0, 8.0]])
out1, W1 = scaled_dot_product_attention(Q1, K1, V1)
assert W1.shape == (1, 1) and np.allclose(W1, 1.0), \
    "a single query/key pair must produce weight exactly 1 (nowhere else to attend)"
assert np.allclose(out1, V1), "with one key, output must equal V exactly"

# Edge case: d_k = 1 (minimal key dimension) still produces a valid probability distribution
Qd = np.array([[1.0], [-1.0]])
Kd = np.array([[1.0], [-1.0], [0.0]])
Vd = np.array([[1.0], [2.0], [3.0]])
_, Wd = scaled_dot_product_attention(Qd, Kd, Vd)
assert np.allclose(Wd.sum(axis=1), 1.0, atol=1e-6), "softmax rows must sum to 1 even with d_k=1"

print("✅ Exercise 1 passed")

<details>
<summary>💡 Show solution</summary>

```python
def scaled_dot_product_attention(Q, K, V):
    d_k = Q.shape[-1]
    scores = Q @ K.T / np.sqrt(d_k)
    scores -= scores.max(axis=-1, keepdims=True)   # numerical stability
    weights = np.exp(scores) / np.exp(scores).sum(axis=-1, keepdims=True)
    return weights @ V, weights
```

</details>

### Exercise 2 — Causal masking

Autoregressive models must not attend to future tokens. A lower-triangular boolean mask
sets future scores to $-\\infty$, so their softmax weight is exactly 0.

Build the mask for a sequence of length $T$, apply it, and verify the
upper triangle of the weight matrix is zero.

In [ ]:
def causal_attention(Q, K, V):
    """Scaled dot-product attention with a causal (lower-triangular) mask."""
    T = Q.shape[0]
    # TODO(you): build the boolean mask (True = allowed) and call
    #            scaled_dot_product_attention with scores masked to -1e9 where False
    ...

In [ ]:
T = 5
d_k = 4
rng2 = np.random.RandomState(7)
Q2 = rng2.randn(T, d_k); K2 = rng2.randn(T, d_k); V2 = rng2.randn(T, d_k)
out2, W2 = causal_attention(Q2, K2, V2)

upper = W2[np.triu_indices(T, k=1)]
assert np.allclose(upper, 0.0, atol=1e-6), \
    "upper triangle of the weight matrix must be 0 (future tokens are masked)"
assert np.allclose(W2.sum(axis=1), 1.0, atol=1e-6), \
    "each row still sums to 1 after masking"
assert W2[0, 0] > 0.99, \
    "first token can only attend to itself, so W[0,0] must be ~1"

# Edge case: single-token sequence -- the causal mask is trivially all-True (only self)
Q1c = np.random.RandomState(3).randn(1, 4)
out1c, W1c = causal_attention(Q1c, Q1c, Q1c)
assert W1c.shape == (1, 1) and np.allclose(W1c, 1.0), \
    "with only one token, the causal mask allows just the diagonal entry"

# Edge case: row i must have exactly i+1 nonzero (allowed) attention weights
T_big = 8
Qb = np.random.RandomState(9).randn(T_big, 3)
_, Wb = causal_attention(Qb, Qb, Qb)
allowed_counts = (Wb > 0).sum(axis=1)
assert list(allowed_counts) == list(range(1, T_big + 1)), \
    "row i must have exactly i+1 nonzero (allowed) attention weights"

print("✅ Exercise 2 passed")

<details>
<summary>💡 Show solution</summary>

```python
def causal_attention(Q, K, V):
    T, d_k = Q.shape
    mask = np.tril(np.ones((T, T))).astype(bool)   # True = allowed
    scores = Q @ K.T / np.sqrt(d_k)
    scores = np.where(mask, scores, -1e9)
    scores -= scores.max(axis=-1, keepdims=True)
    weights = np.exp(scores) / np.exp(scores).sum(axis=-1, keepdims=True)
    return weights @ V, weights
```

</details>

### Exercise 3 — DML practice: `self_attention` (DML #53) and `masked_attention` (DML #107)

Two problems from [Open-Deep-ML](https://github.com/Open-Deep-ML/DML-OpenProblem) that match this
lesson almost exactly, but with slightly different public signatures than the exercises above:

- **DML #53** `self_attention(Q, K, V)` is the same scaled dot-product attention as Exercise 1,
  but returns **only** the output array (no weights tuple).
- **DML #107** `masked_attention(Q, K, V, mask)` is causal-style masking like Exercise 2, but
  `mask` is **additive** (`0` where allowed, `-inf` where disallowed) instead of the boolean
  mask used above -- the two conventions are equivalent, just applied differently.

In [ ]:
def compute_qkv(X, W_q, W_k, W_v):
    """DML #53/#107 helper: project X into Q, K, V."""
    return X @ W_q, X @ W_k, X @ W_v

def self_attention(Q, K, V):
    """DML #53: scaled dot-product self-attention. Returns ONLY the output array
    (unlike Exercise 1's scaled_dot_product_attention, which also returns the weights)."""
    # TODO(you): same math as Exercise 1, but return only the output array
    ...

def masked_attention(Q, K, V, mask):
    """DML #107: masked self-attention with an ADDITIVE mask
    (0 where allowed, -inf where disallowed) -- not the boolean mask of Exercise 2."""
    # TODO(you): scores = QK^T/sqrt(d_k) + mask, then softmax, then weighted sum
    ...

In [ ]:
import numpy as np

# DML #53 -- exact test vectors from tests.json
X = np.array([[1, 0], [0, 1]])
W_q = np.array([[1, 0], [0, 1]]); W_k = np.array([[1, 0], [0, 1]]); W_v = np.array([[1, 2], [3, 4]])
Q, K, V = compute_qkv(X, W_q, W_k, W_v)
out53 = self_attention(Q, K, V)
assert np.allclose(out53, [[1.660477, 2.660477], [2.339523, 3.339523]], atol=1e-5)

X2 = np.array([[1, 0, 1], [0, 1, 1], [1, 1, 0]])
Wq2 = Wk2 = np.array([[1, 1, 0], [0, 1, 1], [1, 0, 1]])
Wv2 = np.array([[1, 2, 3], [4, 5, 6], [7, 8, 9]])
Q2, K2, V2 = compute_qkv(X2, Wq2, Wk2, Wv2)
out53b = self_attention(Q2, K2, V2)
expected53b = [[8.0, 10.0, 12.0], [8.61987385, 10.61987385, 12.61987385], [7.38012615, 9.38012615, 11.38012615]]
assert np.allclose(out53b, expected53b, atol=1e-5)

# Edge case: single-token sequence -- a lone token must attend fully to itself
Xs = np.array([[2.0, -1.0]])
Qs, Ks, Vs = compute_qkv(Xs, np.eye(2), np.eye(2), np.eye(2))
assert np.allclose(self_attention(Qs, Ks, Vs), Xs), "a lone token must attend fully to itself"

# DML #107 -- exact test vector from tests.json (additive causal mask)
np.random.seed(42)
Xm = np.arange(16).reshape(4, 4)
Xm = np.random.permutation(Xm.flatten()).reshape(4, 4)
mask4 = np.triu(np.ones((4, 4)) * (-np.inf), k=1)
Wq4 = np.random.randint(0, 4, size=(4, 4)); Wk4 = np.random.randint(0, 5, size=(4, 4)); Wv4 = np.random.randint(0, 6, size=(4, 4))
Qm, Km, Vm = compute_qkv(Xm, Wq4, Wk4, Wv4)
out107 = masked_attention(Qm, Km, Vm, mask4)
expected107 = [[52.0, 63.0, 48.0, 71.0], [103.0, 109.0, 46.0, 99.0], [103.0, 109.0, 46.0, 99.0], [103.0, 109.0, 46.0, 99.0]]
assert np.allclose(out107, expected107, atol=1e-4)

# Edge case: single-token sequence with a trivial 1x1 additive mask
Q1 = np.array([[1.0, 2.0]]); K1 = Q1.copy(); V1 = np.array([[9.0, 9.0]])
out_single = masked_attention(Q1, K1, V1, np.zeros((1, 1)))
assert np.allclose(out_single, V1), "single-token masked attention must equal V exactly"

# Edge case: an entirely-masked row is mathematically undefined (0/0 in softmax) -> NaN, not a bug
with np.errstate(invalid="ignore"):
    row_mask = np.array([[-np.inf, -np.inf]])
    Qz = np.zeros((1, 2)); Kz = np.zeros((2, 2)); Vz = np.array([[1.0, 1.0], [2.0, 2.0]])
    out_allmasked = masked_attention(Qz, Kz, Vz, row_mask)
assert np.isnan(out_allmasked).all(), \
    "an entirely-masked row has no valid attention distribution (NaN is expected, not an error)"

print("✅ Exercise 3 passed (DML #53 self-attention & #107 masked self-attention)")

<details>
<summary>💡 Show solution</summary>

```python
def compute_qkv(X, W_q, W_k, W_v):
    return X @ W_q, X @ W_k, X @ W_v

def self_attention(Q, K, V):
    d_k = Q.shape[-1]
    scores = Q @ K.T / np.sqrt(d_k)
    scores = scores - scores.max(axis=-1, keepdims=True)
    weights = np.exp(scores) / np.exp(scores).sum(axis=-1, keepdims=True)
    return weights @ V

def masked_attention(Q, K, V, mask):
    d_k = Q.shape[-1]
    scores = Q @ K.T / np.sqrt(d_k) + mask
    scores = scores - scores.max(axis=-1, keepdims=True)
    weights = np.exp(scores) / np.exp(scores).sum(axis=-1, keepdims=True)
    return weights @ V
```

</details>